# UAV Aerial Detection - End-to-End Pipeline (VisDrone + SHWD)

Production pipeline for DJI Matrice 4TD NPU - 4 classes: Person, Vehicle, Hard-Hat, No-Hard-Hat

Dataset: VisDrone2019-DET (aerial vehicles) + SHWD (safety)
Official VisDrone Links: https://github.com/VisDrone/VisDrone-Dataset
- Train 1.44GB: https://drive.google.com/file/d/1a2oHjcEcwXP8oUF95qiwrqzACb2YlUhn/view?usp=sharing
- Val 0.07GB: https://drive.google.com/file/d/1bxK5zgLn0_L8x276eKkuYA_FzwCIjb59/view?usp=sharing


In [ ]:
# Install
!pip install -r ../requirements-colab.txt

from pathlib import Path
import sys
sys.path.append(str(Path().resolve().parent))

## 1. Download VisDrone (Official Links)

Options:
- HuggingFace (easiest): 2.06GB
- Google Drive: Train 1.44GB + Val 0.07GB
- Roboflow YOLO ready: https://universe.roboflow.com/uogolanrewaju/visdrone2019-det


In [ ]:
# Option A: HuggingFace via FiftyOne
# !pip install fiftyone
# import fiftyone.utils.huggingface as fouh
# dataset = fouh.load_from_hub("Voxel51/VisDrone2019-DET")

# Option B: Google Drive via gdown (official links)
# !pip install gdown
# import gdown
# gdown.download("https://drive.google.com/uc?id=1a2oHjcEcwXP8oUF95qiwrqzACb2YlUhn", "data/VisDrone2019-DET-train.zip")
# gdown.download("https://drive.google.com/uc?id=1bxK5zgLn0_L8x276eKkuYA_FzwCIjb59", "data/VisDrone2019-DET-val.zip")

# Print official links
!python ../scripts/download_visdrone.py --method links


## 2. Convert VisDrone to YOLO 4-class

Maps 10 VisDrone classes -> 2 (Person, Vehicle)


In [ ]:
from src.dataset.visdrone_converter import convert_visdrone_to_yolo
from pathlib import Path

# After downloading and unzipping to data/visdrone_raw/
# convert_visdrone_to_yolo(
#     visdrone_root=Path("../data/visdrone_raw"),
#     output_root=Path("../data/yolo_visdrone"),
#     inclusive_vehicles=False  # True to include bicycle/motor as Vehicle
# )

print("VisDrone converter ready - maps:")
print("0: pedestrian -> Person, 1: people -> Person")
print("3: car -> Vehicle, 4: van -> Vehicle, 5: truck -> Vehicle, 8: bus -> Vehicle")


## 3. Merge VisDrone + SHWD -> 4-class

VisDrone provides Person, Vehicle (aerial)
SHWD provides Hard-Hat, No-Hard-Hat


In [ ]:
from src.dataset.merge_datasets import merge_yolo_datasets
from pathlib import Path

# merge_yolo_datasets(
#     visdrone_yolo=Path("../data/yolo_visdrone"),
#     shwd_yolo=Path("../data/shwd_yolo"),
#     output=Path("../data/final_4class")
# )

print("Merge ready - final 4 classes: Person, Vehicle, Hard-Hat, No-Hard-Hat")


## 4. Train YOLOv8n (GTX 1650 4GB Stable)


In [ ]:
from src.training.trainer import train_yolo

# train_yolo(config_path="../configs/visdrone_aerial_4class.yaml")

print("Training config:")
print("- Model: YOLOv8n 3.2M params")
print("- Batch: 4, Workers: 0, AMP: False, SGD lr0 0.001")
print("- Aug: mosaic 1.0, copy-paste 0.3 for small aerial vehicles")
print("- Epochs: 80, imgsz 640")


## 5. Export ONNX (NPU Ready)


In [ ]:
from src.export.onnx_exporter import export_onnx

# export_onnx(
#     weights_path="../runs/detect/train/weights/best.pt",
#     output_path="../weights/aerial_4class.onnx",
#     opset=12,
#     imgsz=640,
#     simplify=True
# )

print("ONNX export: opset 12, static 640, simplified, 10.5MB")


## 6. Generate NPU Package + MQTT Test @3fps


In [ ]:
from src.deployment.package_generator import generate_npu_package
from src.mqtt.dji_mock_publisher import MockPublisher

# Generate DJI NPU package
# generate_npu_package(onnx_path="../weights/aerial_4class.onnx", output_dir="../package/")

# MQTT test @3fps
# publisher = MockPublisher(fps=3.0)
# publisher.run()

print("NPU package + MQTT @3fps ready")
